# Qwen2.5-VL-7B Hybrid LoRA Diagnostics Notebook

Run this notebook in Colab/GPU. Use the config cell to train or evaluate one backend at a time:

- `ADAPTER_MODE = "zero_shot"`: evaluate base Qwen2.5-VL-7B.
- `ADAPTER_MODE = "shared_doc_text_lora"`: train or evaluate one shared adapter for DocVQA + TextVQA.
- `ADAPTER_MODE = "docvqa_lora_only"`: train or evaluate a DocVQA-only adapter.
- `ADAPTER_MODE = "textvqa_lora_only"`: train or evaluate a TextVQA-only adapter.
- `ADAPTER_MODE = "chart_lora_only"`: evaluate a previously saved ChartQA adapter. Train ChartQA adapters in `qwen2vl_lora_rank_sweep.ipynb`.
- `ADAPTER_MODE = "hybrid"`: route ChartQA to the saved `chart_lora` and DocVQA/TextVQA to `shared_doc_text_lora`.

This notebook does not add GRPO, DPO, MoME, teacher distillation, router training, or a new backbone. Current defaults run a ChartQA-only DoRA sanity trial.


## 1. Colab Repository Setup

Run this first in Colab. It clones or updates the repository and changes the working directory to the repo root.

In [ ]:
from pathlib import Path
from collections import Counter
import os
import subprocess

REPO_URL = "https://github.com/kdnehihi/multi-task-moe-vlm-assistant.git"
COLAB_REPO_DIR = Path("/content/multi-task-moe-vlm-assistant")

if Path("/content").exists():
    if not COLAB_REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(COLAB_REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "pull", "origin", "main"], check=True, cwd=COLAB_REPO_DIR)
    os.chdir(COLAB_REPO_DIR)

print(f"Current working directory: {Path.cwd()}")

## 2. Setup

In [ ]:
from pathlib import Path
import contextlib
import gc
import inspect
import json
import os
import random
import re
import subprocess
import sys

os.environ.setdefault("CUDA_LAUNCH_BLOCKING", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT))

required_packages = {
    "qwen-vl-utils": "qwen_vl_utils",
    "accelerate": "accelerate",
    "datasets": "datasets",
    "Pillow": "PIL",
    "scikit-learn": "sklearn",
}
for package_name, import_name in required_packages.items():
    try:
        __import__(import_name)
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package_name], check=True)

import torch

# Colab may preinstall an old torchao version. Recent PEFT checks torchao when it
# exists, even though this notebook does not use torchao quantization.
try:
    import torchao
    torchao_version = getattr(torchao, "__version__", "0.0.0")
    major, minor, *_ = [int(part) for part in torchao_version.split(".")[:2]]
    if (major, minor) < (0, 16):
        print(f"Removing incompatible torchao=={torchao_version} before importing PEFT.")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
        for module_name in list(sys.modules):
            if module_name == "torchao" or module_name.startswith("torchao."):
                del sys.modules[module_name]
except ModuleNotFoundError:
    pass

try:
    import peft
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)

from peft import LoraConfig, PeftModel, get_peft_model
from PIL import Image
from qwen_vl_utils import process_vision_info
from torch.utils.data import DataLoader, Dataset
from transformers import AutoProcessor

from src.data.answers import canonicalize_task_type, choose_training_answer
from src.data.chartqa_sampling import (
    DEFAULT_CHARTQA_2K_QUOTAS,
    attach_chartqa_metadata,
    chartqa_sampling_stats,
    print_chartqa_sampling_debug,
    save_chartqa_sampling_stats,
    stratified_chartqa_sample,
)
from src.evaluation.evaluator import build_prediction_records, summarize_quality_records_by_task


def get_qwen_vl_model_class(model_name: str):
    """Return the right Transformers class for Qwen2-VL or Qwen2.5-VL."""
    if "qwen2.5" in model_name.lower():
        try:
            from transformers import Qwen2_5_VLForConditionalGeneration
        except ImportError as error:
            raise ImportError(
                "Qwen2.5-VL requires a recent transformers version with "
                "Qwen2_5_VLForConditionalGeneration. Upgrade transformers."
            ) from error
        return Qwen2_5_VLForConditionalGeneration

    from transformers import Qwen2VLForConditionalGeneration
    return Qwen2VLForConditionalGeneration

torch.__version__


## 3. Config

These settings drive the full one-pass run. Change limits here only if you want a smaller debug run before the real sanity check.


In [ ]:
METADATA_PATH = PROJECT_ROOT / "data/processed/multitask/validation.jsonl"
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
MODEL_TAG = "qwen25vl_7b"
LOCAL_CHECKPOINT_ROOT = PROJECT_ROOT / "outputs/checkpoints/qwen2vl" / MODEL_TAG
DRIVE_CHECKPOINT_ROOT = Path("/content/drive/MyDrive/multi-task-moe-vlm-assistant/checkpoints/qwen2vl") / MODEL_TAG
USE_GOOGLE_DRIVE_CHECKPOINT = True

# Main flow: run "zero_shot", train "shared_doc_text_lora", then run "hybrid".
# ChartQA adapters should come from qwen2vl_lora_rank_sweep.ipynb, not this notebook.
# Optional modes kept for compatibility: "shared_lora_all_tasks", "shared_lora",
# "chart_lora_only", or "debug_overfit". Use doc/text-only modes to train
# separate adapters without touching the saved ChartQA adapter.
ADAPTER_MODE = "docvqa_lora_only"
TRAIN_ADAPTER = True
TRAIN_TASKS = None
ADAPTER_NAME = None
ALLOW_BASE_FALLBACK = True
USE_BASE_FOR_CHARTQA_FALLBACK = False
# Saved ChartQA LoRA to use during chart_lora_only/hybrid eval.
# Keep None to prefer CHECKPOINT_ROOT/chart_lora, then fall back to the rank-sweep output below.
CHART_LORA_CHECKPOINT_OVERRIDE = None
RANK_SWEEP_MODEL_TAG = "qwen25vl_7b_qlora"
CHART_LORA_SWEEP_CONFIG_NAME = "chart_r8_a16_B_lr2e-5_old_good"
DEBUG_OVERFIT_N = 50

# Default DoRA trial: natural ChartQA eval plus a balanced 2k ChartQA train subset.
USE_FULL_DATA = False
MAX_SAMPLES = 2000
TRAIN_RATIO = 0.8
TEST_LIMIT = None
PRINT_LIMIT = 10
BATCH_SIZE = 4
EPOCHS = 1
LEARNING_RATE = 1e-5
GRADIENT_ACCUMULATION_STEPS = 2
SEED = 42
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 384 * 28 * 28

# Shared DocVQA/TextVQA LoRA defaults.
LORA_R = 4
LORA_ALPHA = 8
LORA_DROPOUT = 0.05
USE_DORA = False
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

# Chart LoRA grid support. Select one config at a time; do not run the full grid by default.
CHART_LORA_MODULE_PRESETS = {
    "A": ["q_proj", "v_proj"],
    "B": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "C": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
}
CHART_LORA_GRID_CONFIG = {
    "rank": 8,
    "lora_alpha": 16,
    "learning_rate": 2e-5,
    "target_modules_preset": "B",
}

# Optional one-at-a-time DoRA sweep. Set RUN_SWEEP_CONFIG=False to use manual config above.
RUN_SWEEP_CONFIG = True
SWEEP_TASK = "docvqa"  # "chartqa", "docvqa", or "textvqa"
SWEEP_CONFIG_INDEX = 0

TASK_DORA_SWEEP_CONFIGS = {
    "chartqa": [
        {"name": "r8_a16_B_lr2e-5", "rank": 8, "alpha": 16, "lr": 2e-5, "preset": "B"},
        {"name": "r16_a32_B_lr5e-6", "rank": 16, "alpha": 32, "lr": 5e-6, "preset": "B"},
        {"name": "r16_a32_C_lr5e-6", "rank": 16, "alpha": 32, "lr": 5e-6, "preset": "C"},
    ],
    "docvqa": [
        {"name": "r4_a8_lr5e-6", "rank": 4, "alpha": 8, "lr": 5e-6},
        {"name": "r8_a16_lr5e-6", "rank": 8, "alpha": 16, "lr": 5e-6},
        {"name": "r8_a16_lr1e-5", "rank": 8, "alpha": 16, "lr": 1e-5},
        {"name": "r16_a32_lr5e-6", "rank": 16, "alpha": 32, "lr": 5e-6},
    ],
    "textvqa": [
        {"name": "r4_a8_lr2e-6", "rank": 4, "alpha": 8, "lr": 2e-6},
        {"name": "r4_a8_lr5e-6", "rank": 4, "alpha": 8, "lr": 5e-6},
        {"name": "r8_a16_lr5e-6", "rank": 8, "alpha": 16, "lr": 5e-6},
        {"name": "r16_a32_lr5e-6", "rank": 16, "alpha": 32, "lr": 5e-6},
    ],
}

CUSTOM_METHOD_NAME = None
if RUN_SWEEP_CONFIG:
    sweep_config = TASK_DORA_SWEEP_CONFIGS[SWEEP_TASK][SWEEP_CONFIG_INDEX]
    USE_DORA = True
    LORA_R = sweep_config["rank"]
    LORA_ALPHA = sweep_config["alpha"]
    LEARNING_RATE = sweep_config["lr"]
    if SWEEP_TASK == "chartqa":
        ADAPTER_MODE = "chart_lora_only"
        ADAPTER_NAME = f"chart_dora_{sweep_config['name']}"
        CHART_LORA_GRID_CONFIG = {
            "rank": sweep_config["rank"],
            "lora_alpha": sweep_config["alpha"],
            "learning_rate": sweep_config["lr"],
            "target_modules_preset": sweep_config.get("preset", "B"),
        }
    elif SWEEP_TASK == "docvqa":
        ADAPTER_MODE = "docvqa_lora_only"
        ADAPTER_NAME = f"docvqa_dora_{sweep_config['name']}"
    elif SWEEP_TASK == "textvqa":
        ADAPTER_MODE = "textvqa_lora_only"
        ADAPTER_NAME = f"textvqa_dora_{sweep_config['name']}"
    else:
        raise ValueError(f"Unsupported SWEEP_TASK: {SWEEP_TASK}")
    CUSTOM_METHOD_NAME = ADAPTER_NAME
    print("Selected DoRA sweep config:", {"task": SWEEP_TASK, **sweep_config, "adapter_name": ADAPTER_NAME})
VALIDATE_DURING_TRAINING = True
CHART_VALIDATION_LIMIT = 60

CHARTQA_BALANCED_SAMPLING = True
CHARTQA_SAMPLE_LIMIT = 2000
CHARTQA_SAMPLING_SEED = 42
CHARTQA_QUOTAS = DEFAULT_CHARTQA_2K_QUOTAS
CHARTQA_STATS_PATH = PROJECT_ROOT / "outputs/data_stats/chartqa_balanced_train_stats.json"
CREATE_BALANCED_DIAGNOSTIC_EVAL = False

DATA_LIMITS = {"docvqa": 667, "chartqa": 667, "textvqa": 666}
EXPECTED_TOTAL_SAMPLES = sum(DATA_LIMITS.values())
FORCE_REBUILD_DATA = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_MIXED_PRECISION = DEVICE == "cuda"
TORCH_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
VALID_ADAPTER_MODES = {
    "zero_shot",
    "shared_lora",
    "shared_lora_all_tasks",
    "shared_doc_text_lora",
    "docvqa_lora_only",
    "textvqa_lora_only",
    "chart_lora_only",
    "hybrid",
    "debug_overfit",
}
assert ADAPTER_MODE in VALID_ADAPTER_MODES
assert not (TRAIN_ADAPTER and ADAPTER_MODE in {"zero_shot", "hybrid"}), "Train one concrete adapter, then run hybrid eval separately."
DEVICE, ADAPTER_MODE, TRAIN_ADAPTER, TRAIN_TASKS


## 4. Checkpoint Storage

In [ ]:
CHECKPOINT_ROOT = LOCAL_CHECKPOINT_ROOT

if USE_GOOGLE_DRIVE_CHECKPOINT:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        CHECKPOINT_ROOT = DRIVE_CHECKPOINT_ROOT
    except Exception as error:
        print(f"Google Drive checkpoint storage is unavailable: {error}")
        CHECKPOINT_ROOT = LOCAL_CHECKPOINT_ROOT

ADAPTER_MODE_ALIASES = {"shared_lora": "shared_lora_all_tasks"}
NORMALIZED_ADAPTER_MODE = ADAPTER_MODE_ALIASES.get(ADAPTER_MODE, ADAPTER_MODE)
RUN_MODE = NORMALIZED_ADAPTER_MODE

MODE_TO_METHOD_NAME = {
    "zero_shot": "zero_shot",
    "shared_lora_all_tasks": "shared_lora_all_tasks",
    "shared_doc_text_lora": "shared_doc_text_lora",
    "docvqa_lora_only": "docvqa_lora_only",
    "textvqa_lora_only": "textvqa_lora_only",
    "chart_lora_only": "chart_lora_only",
    "hybrid": "hybrid_chart_doc_text",
    "debug_overfit": "debug_overfit",
}
MODE_TO_TRAIN_ADAPTER = {
    "shared_lora_all_tasks": "shared_lora_all_tasks",
    "shared_doc_text_lora": "shared_doc_text_lora",
    "docvqa_lora_only": "docvqa_lora",
    "textvqa_lora_only": "textvqa_lora",
    "chart_lora_only": "chart_lora",
    "debug_overfit": "shared_doc_text_lora",
}
MODE_TO_DEFAULT_TRAIN_TASKS = {
    "shared_lora_all_tasks": ["chartqa", "docvqa", "textvqa"],
    "shared_doc_text_lora": ["docvqa", "textvqa"],
    "docvqa_lora_only": ["docvqa"],
    "textvqa_lora_only": ["textvqa"],
    "chart_lora_only": ["chartqa"],
    "debug_overfit": ["docvqa", "textvqa"],
}

method_name = CUSTOM_METHOD_NAME or MODE_TO_METHOD_NAME[NORMALIZED_ADAPTER_MODE]
if CUSTOM_METHOD_NAME is None and USE_DORA and NORMALIZED_ADAPTER_MODE == "chart_lora_only":
    method_name = "chart_dora_only"
ACTIVE_ADAPTER_NAME = ADAPTER_NAME or MODE_TO_TRAIN_ADAPTER.get(NORMALIZED_ADAPTER_MODE)
if TRAIN_ADAPTER:
    ACTIVE_ADAPTER_NAME = ADAPTER_NAME or MODE_TO_TRAIN_ADAPTER[NORMALIZED_ADAPTER_MODE]
    TRAIN_TASKS = TRAIN_TASKS or MODE_TO_DEFAULT_TRAIN_TASKS[NORMALIZED_ADAPTER_MODE]
TRAIN_TASKS = [canonicalize_task_type(task) for task in (TRAIN_TASKS or [])]


def checkpoint_dir_for_adapter(adapter_name: str) -> Path:
    return CHECKPOINT_ROOT / adapter_name


def checkpoint_dir_for_loaded_adapter(adapter_name: str) -> Path:
    return ADAPTER_CHECKPOINTS.get(adapter_name, checkpoint_dir_for_adapter(adapter_name))


def resolve_path(path_value: str | Path) -> Path:
    path = Path(path_value)
    return path if path.is_absolute() else PROJECT_ROOT / path


def chart_lora_checkpoint_dir() -> Path:
    canonical_path = checkpoint_dir_for_adapter("chart_lora")
    sweep_path = (
        PROJECT_ROOT
        / "outputs/checkpoints/qwen2vl/rank_sweep"
        / RANK_SWEEP_MODEL_TAG
        / CHART_LORA_SWEEP_CONFIG_NAME
    )
    if CHART_LORA_CHECKPOINT_OVERRIDE:
        return resolve_path(CHART_LORA_CHECKPOINT_OVERRIDE)
    if canonical_path.exists():
        return canonical_path
    return sweep_path


def prediction_path_for_method(method: str) -> Path:
    return PROJECT_ROOT / "outputs/predictions" / MODEL_TAG / f"qwen2vl_{method}_quality.jsonl"


def report_path_for_method(method: str) -> Path:
    return PROJECT_ROOT / "outputs/predictions" / MODEL_TAG / f"qwen2vl_{method}_report.json"


ADAPTER_CHECKPOINTS = {
    "shared_lora_all_tasks": checkpoint_dir_for_adapter("shared_lora_all_tasks"),
    "shared_doc_text_lora": checkpoint_dir_for_adapter("shared_doc_text_lora"),
    "docvqa_lora": checkpoint_dir_for_adapter("docvqa_lora"),
    "textvqa_lora": checkpoint_dir_for_adapter("textvqa_lora"),
    "chart_lora": chart_lora_checkpoint_dir(),
    "chart_dora": checkpoint_dir_for_adapter("chart_dora"),
}
CHECKPOINT_DIR = checkpoint_dir_for_adapter(ACTIVE_ADAPTER_NAME) if ACTIVE_ADAPTER_NAME else None
active_predictions_path = prediction_path_for_method(method_name)
active_report_path = report_path_for_method(method_name)

print(f"MODEL_NAME: {MODEL_NAME}")
print(f"MODEL_TAG: {MODEL_TAG}")
print(f"Checkpoint root: {CHECKPOINT_ROOT}")
print("Adapter checkpoints:")
for name, path in ADAPTER_CHECKPOINTS.items():
    print(f"  {name}: {path}")
print("Method:", method_name)
print("Active train adapter:", ACTIVE_ADAPTER_NAME)
print("Train tasks:", TRAIN_TASKS)
print("Use DoRA:", USE_DORA)


## 5. Prepare Data If Needed

In [ ]:
def count_jsonl_lines(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for _ in f)


def run_checked_command(command: list[str]) -> None:
    print("Running:", " ".join(command))
    result = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        raise subprocess.CalledProcessError(
            result.returncode,
            command,
            output=result.stdout,
            stderr=result.stderr,
        )
    if result.stderr:
        print(result.stderr)


def effective_data_limits() -> dict[str, int | None]:
    limits = dict(DATA_LIMITS)
    if CHARTQA_BALANCED_SAMPLING and TRAIN_ADAPTER and "chartqa" in set(TRAIN_TASKS):
        required_chart_total = int(CHARTQA_SAMPLE_LIMIT / TRAIN_RATIO + 0.9999)
        limits["chartqa"] = max(limits.get("chartqa") or 0, required_chart_total)
    return limits


EFFECTIVE_DATA_LIMITS = effective_data_limits()
EXPECTED_TOTAL_SAMPLES = sum(limit for limit in EFFECTIVE_DATA_LIMITS.values() if limit is not None)

current_count = count_jsonl_lines(METADATA_PATH)
print(f"Current processed examples: {current_count}")
print("Effective data limits:", EFFECTIVE_DATA_LIMITS)

expected_ready = EXPECTED_TOTAL_SAMPLES is not None and current_count == EXPECTED_TOTAL_SAMPLES

if FORCE_REBUILD_DATA or not expected_ready:
    for dataset_name, limit in EFFECTIVE_DATA_LIMITS.items():
        command = [
            sys.executable,
            str(PROJECT_ROOT / "scripts/prepare_data.py"),
            "--dataset",
            dataset_name,
            "--split",
            "validation",
            "--streaming",
        ]
        if limit is not None:
            command.extend(["--limit", str(limit)])
        run_checked_command(command)

    build_command = [sys.executable, str(PROJECT_ROOT / "scripts/build_multitask_data.py"), "--split", "validation"]
    run_checked_command(build_command)
else:
    print(f"Processed data already has the expected {EXPECTED_TOTAL_SAMPLES}-example size. Skipping data preparation.")


## 6. Load Records

In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)


def canonical_task_type(record: dict) -> str:
    return canonicalize_task_type(record.get("task_type"), record.get("dataset")) or "textvqa"


all_records = []
with METADATA_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        record["canonical_task_type"] = canonical_task_type(record)
        record["task_type"] = record["canonical_task_type"]
        record["chosen_training_answer"] = choose_training_answer(
            record.get("answers", []),
            record["canonical_task_type"],
        )
        if record["canonical_task_type"] == "chartqa":
            attach_chartqa_metadata(record)
        all_records.append(record)

TASK_TYPES = ("chartqa", "docvqa", "textvqa")
all_records_by_task = {
    task_type: [record for record in all_records if record["canonical_task_type"] == task_type]
    for task_type in TASK_TYPES
}
for task_records in all_records_by_task.values():
    random.shuffle(task_records)

def balanced_task_limits(total: int, task_types: tuple[str, ...]) -> dict[str, int]:
    base = total // len(task_types)
    remainder = total % len(task_types)
    return {
        task_type: base + (1 if index < remainder else 0)
        for index, task_type in enumerate(task_types)
    }


use_balanced_chartqa_train = (
    CHARTQA_BALANCED_SAMPLING
    and TRAIN_ADAPTER
    and "chartqa" in set(TRAIN_TASKS)
)

if USE_FULL_DATA:
    records = list(all_records)
else:
    if MAX_SAMPLES is None:
        raise ValueError("MAX_SAMPLES must be set when USE_FULL_DATA=False.")
    limits = balanced_task_limits(MAX_SAMPLES, TASK_TYPES)
    if use_balanced_chartqa_train:
        required_chart_total = int(CHARTQA_SAMPLE_LIMIT / TRAIN_RATIO + 0.9999)
        limits["chartqa"] = max(limits["chartqa"], required_chart_total)
    records = []
    for task_type in TASK_TYPES:
        available = all_records_by_task[task_type]
        requested = limits[task_type]
        if len(available) < requested:
            raise ValueError(f"Need {requested} {task_type} records, found {len(available)}.")
        records.extend(available[:requested])

# Split inside each task first, then merge. This preserves a natural/fixed eval
# set; balanced ChartQA sampling is applied only to the training side below.
train_records = []
test_records = []
records_by_selected_task = {
    task_type: [record for record in records if record["canonical_task_type"] == task_type]
    for task_type in TASK_TYPES
}
for task_type, task_records in records_by_selected_task.items():
    random.shuffle(task_records)
    split_index = int(len(task_records) * TRAIN_RATIO)
    train_records.extend(task_records[:split_index])
    test_records.extend(task_records[split_index:])

chartqa_sampling_report = None
if use_balanced_chartqa_train:
    non_chart_train_records = [record for record in train_records if record["canonical_task_type"] != "chartqa"]
    chart_train_pool = [record for record in train_records if record["canonical_task_type"] == "chartqa"]
    balanced_chart_train_records = stratified_chartqa_sample(
        chart_train_pool,
        CHARTQA_QUOTAS,
        seed=CHARTQA_SAMPLING_SEED,
        sample_limit=CHARTQA_SAMPLE_LIMIT,
    )
    chartqa_sampling_report = chartqa_sampling_stats(
        balanced_chart_train_records,
        CHARTQA_QUOTAS,
        CHARTQA_SAMPLING_SEED,
        CHARTQA_SAMPLE_LIMIT,
    )
    save_chartqa_sampling_stats(chartqa_sampling_report, CHARTQA_STATS_PATH)
    print("ChartQA balanced train stats:", json.dumps(chartqa_sampling_report, indent=2))
    print_chartqa_sampling_debug(balanced_chart_train_records)
    train_records = non_chart_train_records + balanced_chart_train_records

random.shuffle(train_records)
random.shuffle(test_records)
records = train_records + test_records

def eval_tasks_for_mode(mode: str) -> set[str] | None:
    if mode in {"shared_doc_text_lora", "debug_overfit"}:
        return {"docvqa", "textvqa"}
    if mode == "docvqa_lora_only":
        return {"docvqa"}
    if mode == "textvqa_lora_only":
        return {"textvqa"}
    if mode == "chart_lora_only":
        return {"chartqa"}
    return None


def filter_records_by_tasks(source_records: list[dict], task_filter: set[str] | None) -> list[dict]:
    if task_filter is None:
        return list(source_records)
    return [record for record in source_records if record["canonical_task_type"] in task_filter]


if NORMALIZED_ADAPTER_MODE == "debug_overfit":
    candidate_train_records = [record for record in train_records if record["canonical_task_type"] in set(TRAIN_TASKS)]
    active_train_records = candidate_train_records[:DEBUG_OVERFIT_N]
    eval_records = active_train_records
elif TRAIN_ADAPTER:
    active_train_records = [record for record in train_records if record["canonical_task_type"] in set(TRAIN_TASKS)]
    mode_eval_records = filter_records_by_tasks(test_records, eval_tasks_for_mode(NORMALIZED_ADAPTER_MODE))
    eval_records = mode_eval_records if TEST_LIMIT is None else mode_eval_records[:TEST_LIMIT]
else:
    active_train_records = []
    mode_eval_records = filter_records_by_tasks(test_records, eval_tasks_for_mode(NORMALIZED_ADAPTER_MODE))
    eval_records = mode_eval_records if TEST_LIMIT is None else mode_eval_records[:TEST_LIMIT]

chart_validation_records = [record for record in test_records if record["canonical_task_type"] == "chartqa"][:CHART_VALIDATION_LIMIT]

print("Total selected records:", len(records), Counter(record["canonical_task_type"] for record in records))
print("Train records:", len(train_records), Counter(record["canonical_task_type"] for record in train_records))
print("Test records:", len(test_records), Counter(record["canonical_task_type"] for record in test_records))
print("Active train records:", len(active_train_records), Counter(record["canonical_task_type"] for record in active_train_records))
if chartqa_sampling_report:
    print("ChartQA train question types:", chartqa_sampling_report["count_by_question_type"])
    print("ChartQA train answer types:", chartqa_sampling_report["count_by_answer_type"])
    print(f"Saved ChartQA sampling stats to {CHARTQA_STATS_PATH}")
print("Eval records:", len(eval_records), Counter(record["canonical_task_type"] for record in eval_records))
print("Chart validation records:", len(chart_validation_records))
print("Method:", method_name)

records[0]


# One-Pass Run: Shared LoRA Train and Evaluation


## 7. Load Processor For Training Batches


In [ ]:
def free_gpu_memory() -> None:
    for name in ["model", "optimizer", "scaler"]:
        if name in globals():
            del globals()[name]
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


free_gpu_memory()
processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
print("Processor loaded for shared-LoRA training batches.")


## 8. Prompting, Generation, and Prediction Helpers


In [ ]:
TASK_PROMPTS = {
    "chartqa": """Read the chart carefully.
Use the chart title, axis labels, legend, colors, categories, and values to answer the question.
If the question asks yes/no, answer only Yes or No.
Otherwise return only the final value, label, or short phrase.
Do not explain.
Do not include extra text.

Question: {question}
Answer:""",
    "docvqa": """Read the document and answer the question.
Return only the exact shortest answer span.
Do not include nearby words, headers, titles, or surrounding document text.
Do not explain.

Question: {question}
Answer:""",
    "textvqa": """Read the visible text in the image and answer the question.
Return only the exact shortest answer span.
Do not copy unrelated visible text.
Do not list all visible words.
Do not explain.

Question: {question}
Answer:""",
    "default": """Answer the question using the image.
Return only the shortest answer.
Do not explain.

Question: {question}
Answer:""",
}

TASK_GENERATION_KWARGS = {
    "chartqa": {"max_new_tokens": 8, "do_sample": False, "num_beams": 1, "repetition_penalty": 1.1},
    "docvqa": {"max_new_tokens": 16, "do_sample": False, "num_beams": 1, "repetition_penalty": 1.15},
    "textvqa": {"max_new_tokens": 12, "do_sample": False, "num_beams": 1, "repetition_penalty": 1.15},
    "default": {"max_new_tokens": 12, "do_sample": False, "num_beams": 1, "repetition_penalty": 1.15},
}


def format_question_prompt(question: str, task_type: str | None = None) -> str:
    canonical_task = canonicalize_task_type(task_type) or "default"
    template = TASK_PROMPTS.get(canonical_task, TASK_PROMPTS["default"])
    return template.format(question=question)


def generation_kwargs_for_task(task_type: str | None) -> dict:
    canonical_task = canonicalize_task_type(task_type) or "default"
    return TASK_GENERATION_KWARGS.get(canonical_task, TASK_GENERATION_KWARGS["default"])


YES_NO_QUESTION_RE = re.compile(
    r"^\s*(is|are|was|were|do|does|did|can|could|has|have|had|will|would|"
    r"should|may|might)\b",
    re.IGNORECASE,
)


def is_yes_no_question(question: str) -> bool:
    return bool(YES_NO_QUESTION_RE.search(str(question)))


def clean_generated_answer(answer: str, question: str | None = None) -> str:
    cleaned = " ".join(str(answer).strip().split())
    if not cleaned:
        return cleaned
    if question is not None and not is_yes_no_question(question):
        cleaned = re.sub(r"\s+(?:yes|no)\s*$", "", cleaned, flags=re.IGNORECASE)

    lowered = cleaned.lower()
    if lowered.startswith("unanswerable") or lowered.startswith("not a question"):
        return "unanswerable"

    tokens = cleaned.split()
    for start in range(1, len(tokens)):
        suffix = tokens[start:]
        if len(suffix) < 3:
            continue
        numeric_like = sum(bool(re.fullmatch(r"[\d.,:%$/-]+|[a-z]*\d+[a-z]*", token.lower())) for token in suffix)
        if numeric_like / len(suffix) >= 0.8:
            return " ".join(tokens[:start]).strip(" ,.;:")

    return cleaned.strip(" ,.;:")


def build_messages(image_path: str, question: str, task_type: str | None = None, answer: str | None = None):
    user_content = [
        {"type": "image", "image": str(PROJECT_ROOT / image_path)},
        {"type": "text", "text": format_question_prompt(question, task_type)},
    ]
    messages = [{"role": "user", "content": user_content}]
    if answer is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return messages


def adapter_for_task(task_type: str | None) -> str | None:
    canonical_task = canonicalize_task_type(task_type) or "textvqa"
    if NORMALIZED_ADAPTER_MODE == "zero_shot":
        return None
    if NORMALIZED_ADAPTER_MODE == "shared_lora_all_tasks":
        return "shared_lora_all_tasks"
    if NORMALIZED_ADAPTER_MODE in {"shared_doc_text_lora", "debug_overfit"}:
        return "shared_doc_text_lora" if canonical_task in {"docvqa", "textvqa"} else None
    if NORMALIZED_ADAPTER_MODE == "docvqa_lora_only":
        return ACTIVE_ADAPTER_NAME if canonical_task == "docvqa" else None
    if NORMALIZED_ADAPTER_MODE == "textvqa_lora_only":
        return ACTIVE_ADAPTER_NAME if canonical_task == "textvqa" else None
    if NORMALIZED_ADAPTER_MODE == "chart_lora_only":
        return ACTIVE_ADAPTER_NAME if canonical_task == "chartqa" else None
    if NORMALIZED_ADAPTER_MODE == "hybrid":
        if canonical_task == "chartqa":
            return None if USE_BASE_FOR_CHARTQA_FALLBACK else "chart_lora"
        if canonical_task == "docvqa":
            return "docvqa_lora" if "docvqa_lora" in loaded_adapters else "shared_doc_text_lora"
        if canonical_task == "textvqa":
            return "textvqa_lora" if "textvqa_lora" in loaded_adapters else "shared_doc_text_lora"
    return None


def generate_with_selected_adapter(inputs, selected_adapter: str | None):
    if selected_adapter and hasattr(model, "set_adapter"):
        model.set_adapter(selected_adapter)
        adapter_context = contextlib.nullcontext()
    elif selected_adapter is None and hasattr(model, "disable_adapter"):
        adapter_context = model.disable_adapter()
    else:
        adapter_context = contextlib.nullcontext()

    with adapter_context, torch.inference_mode():
        return model.generate(**inputs, **generation_kwargs_for_task(current_generation_task_type))


def qwen_predict(example: dict, return_raw: bool = False):
    global current_generation_task_type
    task_type = example.get("canonical_task_type") or example.get("task_type")
    current_generation_task_type = task_type
    selected_adapter = adapter_for_task(task_type)
    messages = build_messages(example["image_path"], example["question"], task_type)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to(model.device)

    generated_ids = generate_with_selected_adapter(inputs, selected_adapter)
    generated_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    ]
    raw_prediction = processor.batch_decode(generated_trimmed, skip_special_tokens=True)[0].strip()
    cleaned_prediction = clean_generated_answer(raw_prediction, example.get("question", ""))
    if return_raw:
        return raw_prediction, cleaned_prediction, selected_adapter
    return cleaned_prediction


def evaluate_records(records_to_eval: list[dict], label: str, print_limit: int = 0):
    model.eval()
    prediction_records = []

    for index, example in enumerate(records_to_eval, start=1):
        raw_prediction, cleaned_prediction, selected_adapter = qwen_predict(example, return_raw=True)
        prediction_records.append({
            "index": index,
            "question": example["question"],
            "answers": example["answers"],
            "chosen_training_answer": example.get("chosen_training_answer"),
            "raw_prediction": raw_prediction,
            "cleaned_prediction": cleaned_prediction,
            "true_task": example["canonical_task_type"],
            "adapter_backend": selected_adapter or "base_zero_shot",
        })

        if index <= print_limit:
            print(f"[{index}/{len(records_to_eval)}] {example['dataset']} {example['canonical_task_type']} adapter={selected_adapter or 'base'} -> {cleaned_prediction!r}")
            print("answers:", example["answers"])

    quality_records = build_prediction_records(
        [record["raw_prediction"] for record in prediction_records],
        records_to_eval,
        cleaned_predictions=[record["cleaned_prediction"] for record in prediction_records],
    )
    for quality_record, prediction_record in zip(quality_records, prediction_records):
        quality_record["method"] = label
        quality_record["adapter_backend"] = prediction_record["adapter_backend"]
    quality_report = summarize_quality_records_by_task(quality_records)
    return quality_records, quality_report, prediction_records


## 9. Build Shared-LoRA Fine-Tuning Dataset


## 9. Dataset and Collate Function


In [ ]:
class QwenVQAFinetuneDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        example = self.examples[index]
        task_type = example["canonical_task_type"]
        answer = choose_training_answer(example.get("answers", []), task_type)
        return {
            "image_path": example["image_path"],
            "question": example["question"],
            "answers": example.get("answers", []),
            "answer": answer,
            "task_type": task_type,
        }


def find_subsequence(sequence, subsequence):
    if not subsequence or len(subsequence) > len(sequence):
        return -1
    for start in range(len(sequence) - len(subsequence) + 1):
        if sequence[start:start + len(subsequence)] == subsequence:
            return start
    return -1


def mask_prompt_tokens(labels, full_inputs, prompt_inputs, row_index: int) -> None:
    full_ids = full_inputs["input_ids"][row_index].tolist()
    prompt_ids = prompt_inputs["input_ids"][row_index][prompt_inputs["attention_mask"][row_index].bool()].tolist()
    prompt_start = find_subsequence(full_ids, prompt_ids)
    if prompt_start < 0:
        raise ValueError(f"Could not align prompt tokens for row {row_index}; refusing to train on unsafe labels.")
    labels[row_index, prompt_start:prompt_start + len(prompt_ids)] = -100


def collate_fn(batch):
    full_messages = [build_messages(item["image_path"], item["question"], item["task_type"], item["answer"]) for item in batch]
    prompt_messages = [build_messages(item["image_path"], item["question"], item["task_type"]) for item in batch]
    task_types = [item["task_type"] for item in batch]
    questions = [item["question"] for item in batch]
    answers = [item["answer"] for item in batch]
    reference_answers = [item["answers"] for item in batch]

    full_texts = [processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False) for messages in full_messages]
    prompt_texts = [processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) for messages in prompt_messages]
    image_inputs, video_inputs = process_vision_info(full_messages)

    inputs = processor(
        text=full_texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    prompt_inputs = processor(
        text=prompt_texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    labels = inputs["input_ids"].clone()

    for row_index in range(labels.shape[0]):
        mask_prompt_tokens(labels, inputs, prompt_inputs, row_index)

    pad_token_id = processor.tokenizer.pad_token_id
    if pad_token_id is not None:
        labels[labels == pad_token_id] = -100

    special_token_ids = set(processor.tokenizer.all_special_ids)
    for token_id in special_token_ids:
        labels[labels == token_id] = -100
    inputs["labels"] = labels
    inputs["task_types"] = task_types
    inputs["questions"] = questions
    inputs["chosen_training_answers"] = answers
    inputs["reference_answers"] = reference_answers
    return inputs


if TRAIN_ADAPTER:
    train_dataset = QwenVQAFinetuneDataset(active_train_records)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    print("Train dataset examples:", len(train_dataset))
else:
    train_dataset = None
    train_loader = None
    print("Evaluation-only mode: skipping SFT dataset/loader.")


## 10. Decode Labels Before Training

This cell must pass before training starts. It decodes the first batch labels and asserts that labels contain only the assistant answer.


In [ ]:
def debug_decode_labels(processor, batch, n=5):
    labels = batch["labels"].detach().cpu().clone()
    pad_id = processor.tokenizer.pad_token_id
    if pad_id is None:
        pad_id = processor.tokenizer.eos_token_id
    pad_text = processor.tokenizer.pad_token or processor.tokenizer.eos_token or ""

    decoded_rows = []
    for i in range(min(n, labels.shape[0])):
        row = labels[i]
        row[row == -100] = pad_id
        decoded = processor.tokenizer.decode(row, skip_special_tokens=False)
        decoded_label_text = " ".join(decoded.replace(pad_text, " ").split()) if pad_text else " ".join(decoded.split())
        expected = batch["chosen_training_answers"][i]
        question = batch["questions"][i]
        print(f"[task_type {i}] {batch['task_types'][i]}")
        print(f"[raw question {i}] {question!r}")
        print(f"[raw answers {i}] {batch['reference_answers'][i]!r}")
        print(f"[chosen training answer {i}] {expected!r}")
        print(f"[decoded labels {i}] {decoded_label_text!r}")
        decoded_rows.append(decoded_label_text)

        assert expected in decoded_label_text, f"Expected answer missing from decoded labels row {i}: {expected!r}"
        assert question not in decoded_label_text, f"Question leaked into labels row {i}."
        forbidden_fragments = ["<|im_start|>", "<|im_end|>", "<|vision_start|>", "<|vision_end|>", "<|image_pad|>", "assistant"]
        leaked_tokens = [token for token in forbidden_fragments if token in decoded_label_text]
        assert not leaked_tokens, f"Special/image/chat tokens leaked into labels row {i}: {leaked_tokens}"
        assert decoded_label_text == expected, f"Labels should contain only the assistant answer for row {i}: {decoded_label_text!r} != {expected!r}"
    return decoded_rows


if TRAIN_ADAPTER:
    debug_batch = next(iter(train_loader))
    for key, value in debug_batch.items():
        if hasattr(value, "shape"):
            print(key, value.shape, value.dtype)
        else:
            print(key, value)

    valid_labels = debug_batch["labels"][debug_batch["labels"] != -100]
    print("label min/max:", int(valid_labels.min()), int(valid_labels.max()))
    print("tokenizer vocab size:", processor.tokenizer.vocab_size)
    print("tokenizer length with added/special tokens:", len(processor.tokenizer))
    assert int(valid_labels.min()) >= 0
    assert int(valid_labels.max()) < len(processor.tokenizer)
    decoded_label_rows = debug_decode_labels(processor, debug_batch, n=5)
    print("Batch check passed.")
else:
    print("Evaluation-only mode: skipping label sanity check.")


## 11. Attach One Shared LoRA Adapter


In [ ]:
free_gpu_memory()
processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
model_class = get_qwen_vl_model_class(MODEL_NAME)
model = model_class.from_pretrained(
    MODEL_NAME,
    torch_dtype=TORCH_DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
)
if DEVICE != "cuda":
    model.to(DEVICE)

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

loaded_adapters = set()


def active_lora_hparams() -> dict:
    if ACTIVE_ADAPTER_NAME == "chart_lora":
        preset_name = CHART_LORA_GRID_CONFIG["target_modules_preset"]
        return {
            "r": CHART_LORA_GRID_CONFIG["rank"],
            "lora_alpha": CHART_LORA_GRID_CONFIG["lora_alpha"],
            "target_modules": CHART_LORA_MODULE_PRESETS[preset_name],
            "learning_rate": CHART_LORA_GRID_CONFIG["learning_rate"],
            "preset_name": preset_name,
            "use_dora": USE_DORA,
        }
    return {
        "r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "target_modules": LORA_TARGET_MODULES,
        "learning_rate": LEARNING_RATE,
        "preset_name": "shared_default",
        "use_dora": USE_DORA,
    }


def required_adapters_for_mode() -> list[str]:
    if NORMALIZED_ADAPTER_MODE == "shared_lora_all_tasks":
        return ["shared_lora_all_tasks"]
    if NORMALIZED_ADAPTER_MODE in {"shared_doc_text_lora", "debug_overfit"}:
        return ["shared_doc_text_lora"]
    if NORMALIZED_ADAPTER_MODE == "docvqa_lora_only":
        return [ACTIVE_ADAPTER_NAME or "docvqa_lora"]
    if NORMALIZED_ADAPTER_MODE == "textvqa_lora_only":
        return [ACTIVE_ADAPTER_NAME or "textvqa_lora"]
    if NORMALIZED_ADAPTER_MODE == "chart_lora_only":
        return [ACTIVE_ADAPTER_NAME or "chart_lora"]
    if NORMALIZED_ADAPTER_MODE == "hybrid":
        adapters = ["shared_doc_text_lora"]
        if not USE_BASE_FOR_CHARTQA_FALLBACK:
            adapters.append("chart_lora")
        return adapters
    return []


def resolve_adapter_load_path(adapter_name: str, adapter_path: Path) -> Path:
    direct_config = adapter_path / "adapter_config.json"
    nested_path = adapter_path / adapter_name
    nested_config = nested_path / "adapter_config.json"
    if direct_config.exists():
        return adapter_path
    if nested_config.exists():
        return nested_path
    raise FileNotFoundError(
        f"Missing adapter_config.json for {adapter_name}. Checked {direct_config} and {nested_config}."
    )


if TRAIN_ADAPTER:
    model.config.use_cache = False
    hparams = active_lora_hparams()
    lora_kwargs = {
        "r": hparams["r"],
        "lora_alpha": hparams["lora_alpha"],
        "target_modules": hparams["target_modules"],
        "lora_dropout": LORA_DROPOUT,
        "bias": "none",
        "task_type": "CAUSAL_LM",
    }
    if hparams.get("use_dora"):
        if "use_dora" not in inspect.signature(LoraConfig).parameters:
            raise RuntimeError("This PEFT version does not support DoRA. Run: pip install -U peft")
        lora_kwargs["use_dora"] = True
    lora_config = LoraConfig(**lora_kwargs)
    model = get_peft_model(model, lora_config, adapter_name=ACTIVE_ADAPTER_NAME)
    loaded_adapters.add(ACTIVE_ADAPTER_NAME)
    if DEVICE != "cuda":
        model.to(DEVICE)
    model.train()
    model.print_trainable_parameters()
    optimizer = torch.optim.AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=hparams["learning_rate"],
    )
    print("Training LoRA hparams:", hparams)
else:
    required_adapters = required_adapters_for_mode()
    for adapter_name in required_adapters:
        adapter_path = checkpoint_dir_for_loaded_adapter(adapter_name)
        if not adapter_path.exists():
            if adapter_name == "chart_lora" and ALLOW_BASE_FALLBACK and USE_BASE_FOR_CHARTQA_FALLBACK:
                print(f"Missing {adapter_name} at {adapter_path}; ChartQA will use base fallback.")
                continue
            raise FileNotFoundError(f"Missing adapter checkpoint for {adapter_name}: {adapter_path}")
        load_path = resolve_adapter_load_path(adapter_name, adapter_path)
        if not loaded_adapters:
            model = PeftModel.from_pretrained(model, load_path, adapter_name=adapter_name, is_trainable=False)
        else:
            model.load_adapter(load_path, adapter_name=adapter_name, is_trainable=False)
        loaded_adapters.add(adapter_name)
        print(f"Loaded adapter {adapter_name} from {load_path}")
    model.eval()
    optimizer = None
    print("Evaluation mode loaded adapters:", sorted(loaded_adapters) or "base only")


## 12. Train Shared LoRA


In [ ]:
checkpoint_metrics = []
best_checkpoint_metric = None
best_checkpoint_epoch = None
checkpoint_saved = False

if TRAIN_ADAPTER:
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    autocast_device = "cuda" if DEVICE == "cuda" else "cpu"
    scaler = torch.cuda.amp.GradScaler(enabled=USE_MIXED_PRECISION)
    loss_history = []
    loss_by_task = {"chartqa": [], "docvqa": [], "textvqa": []}
    optimizer.zero_grad(set_to_none=True)

    for epoch in range(EPOCHS):
        model.train()
        for step, batch in enumerate(train_loader, start=1):
            task_types = batch.pop("task_types")
            batch.pop("questions", None)
            batch.pop("chosen_training_answers", None)
            batch.pop("reference_answers", None)
            batch = {key: value.to(model.device) if hasattr(value, "to") else value for key, value in batch.items()}

            with torch.autocast(device_type=autocast_device, dtype=torch.float16, enabled=USE_MIXED_PRECISION):
                outputs = model(**batch)
                loss = outputs.loss

            scaler.scale(loss / GRADIENT_ACCUMULATION_STEPS).backward()

            if step % GRADIENT_ACCUMULATION_STEPS == 0 or step == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            loss_value = float(loss.detach().cpu())
            loss_history.append(loss_value)
            for task_type in task_types:
                loss_by_task.setdefault(task_type, []).append(loss_value)

            if step % 10 == 0 or step == 1:
                print(f"epoch={epoch + 1} step={step} loss={loss_value:.4f}")

        epoch_record = {"epoch": epoch + 1, "adapter_name": ACTIVE_ADAPTER_NAME}
        if ACTIVE_ADAPTER_NAME == "chart_lora" and VALIDATE_DURING_TRAINING and chart_validation_records:
            validation_quality_records, validation_report, _ = evaluate_records(
                chart_validation_records,
                label=f"{method_name}_epoch_{epoch + 1}",
                print_limit=0,
            )
            chart_metric = validation_report["by_task"].get("chartqa", {}).get("chart_hybrid_accuracy") or 0.0
            epoch_record["chart_hybrid_accuracy"] = chart_metric
            epoch_record["validation_report"] = validation_report
            should_save = best_checkpoint_metric is None or chart_metric > best_checkpoint_metric
            print(f"epoch={epoch + 1} chart_hybrid_accuracy={chart_metric:.4f} best={best_checkpoint_metric}")
            if should_save:
                best_checkpoint_metric = chart_metric
                best_checkpoint_epoch = epoch + 1
                CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
                model.save_pretrained(CHECKPOINT_DIR)
                processor.save_pretrained(CHECKPOINT_DIR)
                checkpoint_saved = True
                print(f"Saved best chart_lora checkpoint to {CHECKPOINT_DIR}")
        checkpoint_metrics.append(epoch_record)

    def mean_loss(losses: list[float]) -> float | None:
        if not losses:
            return None
        return sum(losses) / len(losses)


    loss_summary = {
        "chartqa_loss": mean_loss(loss_by_task["chartqa"]),
        "docvqa_loss": mean_loss(loss_by_task["docvqa"]),
        "textvqa_loss": mean_loss(loss_by_task["textvqa"]),
        "overall_loss": mean_loss(loss_history),
        "best_checkpoint_metric": best_checkpoint_metric,
        "best_checkpoint_epoch": best_checkpoint_epoch,
    }

    print("Loss summary:")
    for name, value in loss_summary.items():
        print(f"{name}: {value:.4f}" if isinstance(value, float) else f"{name}: {value}")
else:
    loss_history = []
    loss_summary = None
    print("Evaluation-only mode: skipping training loop.")

loss_summary


## 13. Save Shared LoRA Adapter Checkpoint


In [ ]:
if TRAIN_ADAPTER:
    if not checkpoint_saved:
        CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(CHECKPOINT_DIR)
        processor.save_pretrained(CHECKPOINT_DIR)
        checkpoint_saved = True
        print(f"Saved Qwen2-VL LoRA checkpoint to {CHECKPOINT_DIR}")
    else:
        print(f"Best checkpoint already saved to {CHECKPOINT_DIR}")
else:
    print("Evaluation-only mode: no adapter checkpoint to save.")


## 14. Evaluate Shared LoRA On The Test Split


In [ ]:
quality_records, quality_report, prediction_records = evaluate_records(
    eval_records,
    label=method_name,
    print_limit=PRINT_LIMIT if NORMALIZED_ADAPTER_MODE == "debug_overfit" else PRINT_LIMIT,
)

active_predictions_path.parent.mkdir(parents=True, exist_ok=True)
with active_predictions_path.open("w", encoding="utf-8") as f:
    for record in quality_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"{method_name} evaluated samples:", len(prediction_records))
print(json.dumps(quality_report, indent=2))
print(f"Saved per-sample quality records to {active_predictions_path}")
quality_records[:3]


## 15. Save Shared LoRA Report


In [ ]:
report_bundle = {
    method_name: quality_report,
    "loss_summary": loss_summary,
    "checkpoint_metrics": checkpoint_metrics,
    "config": {
        "adapter_mode": NORMALIZED_ADAPTER_MODE,
        "train_adapter": TRAIN_ADAPTER,
        "train_tasks": TRAIN_TASKS,
        "active_adapter_name": ACTIVE_ADAPTER_NAME,
        "allow_base_fallback": ALLOW_BASE_FALLBACK,
        "use_base_for_chartqa_fallback": USE_BASE_FOR_CHARTQA_FALLBACK,
        "chart_lora_grid_config": CHART_LORA_GRID_CONFIG,
        "use_dora": USE_DORA,
        "run_sweep_config": RUN_SWEEP_CONFIG,
        "sweep_task": SWEEP_TASK if RUN_SWEEP_CONFIG else None,
        "sweep_config_index": SWEEP_CONFIG_INDEX if RUN_SWEEP_CONFIG else None,
        "custom_method_name": CUSTOM_METHOD_NAME,
        "chartqa_balanced_sampling": CHARTQA_BALANCED_SAMPLING,
        "chartqa_sample_limit": CHARTQA_SAMPLE_LIMIT,
        "chartqa_sampling_seed": CHARTQA_SAMPLING_SEED,
        "chartqa_quotas": dict(CHARTQA_QUOTAS),
    },
    "paths": {
        "predictions": str(active_predictions_path),
        "report": str(active_report_path),
        "checkpoint": str(CHECKPOINT_DIR) if CHECKPOINT_DIR else None,
        "shared_doc_text_lora": str(ADAPTER_CHECKPOINTS["shared_doc_text_lora"]),
        "docvqa_lora": str(ADAPTER_CHECKPOINTS["docvqa_lora"]),
        "textvqa_lora": str(ADAPTER_CHECKPOINTS["textvqa_lora"]),
        "chart_lora": str(ADAPTER_CHECKPOINTS["chart_lora"]),
        "shared_lora_all_tasks": str(ADAPTER_CHECKPOINTS["shared_lora_all_tasks"]),
        "chart_dora": str(ADAPTER_CHECKPOINTS["chart_dora"]),
        "chartqa_sampling_stats": str(CHARTQA_STATS_PATH),
    },
}

active_report_path.parent.mkdir(parents=True, exist_ok=True)
active_report_path.write_text(
    json.dumps(report_bundle, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(quality_report, indent=2))
print(f"Saved report to {active_report_path}")
report_bundle


## 16. Quality Diagnostics

Inspect failure modes directly inside the notebook.


In [ ]:
def print_quality_case(record: dict) -> None:
    print(
        f"[{record.get('index')}] task={record.get('task_type')} "
        f"backend={record.get('adapter_backend')} "
        f"len={record.get('output_length')} "
        f"em={record.get('normalized_em')} "
        f"strict={record.get('strict_containment')} "
        f"chart_relaxed={record.get('chart_relaxed_accuracy')} "
        f"chart_hybrid={record.get('chart_hybrid_accuracy')} "
        f"anls={record.get('anls')} "
        f"vqa={record.get('vqa_score')}"
    )
    print("Q:", record.get("question"))
    print("A:", record.get("answers"))
    print("Raw P:", record.get("raw_prediction"))
    print("Clean P:", record.get("cleaned_prediction"))
    print()


def print_quality_section(title: str, records: list[dict], limit: int = 20) -> None:
    print(f"\n## {title} ({len(records)})")
    for record in records[:limit]:
        print_quality_case(record)


old_true_strict_false = [
    record for record in quality_records
    if record.get("old_containment_if_available") == 1.0 and record.get("strict_containment") == 0.0
]
strict_true_em_false = [
    record for record in quality_records
    if record.get("strict_containment") == 1.0 and record.get("normalized_em") == 0.0
]
chart_relaxed_em_false = [
    record for record in quality_records
    if record.get("chart_relaxed_accuracy") == 1.0 and record.get("normalized_em") == 0.0
]
chart_wrong = [
    record for record in quality_records
    if record.get("task_type") == "chartqa" and record.get("chart_hybrid_accuracy") == 0.0
]
long_outputs = [record for record in quality_records if record.get("output_length", 0) > 10]
worst_anls = sorted(
    [record for record in quality_records if record.get("task_type") in {"docvqa", "textvqa"} and record.get("anls") is not None],
    key=lambda record: record.get("anls", 0.0),
)
random.seed(SEED)
random_samples = random.sample(quality_records, min(20, len(quality_records)))

print_quality_section("Old containment true but strict containment false", old_true_strict_false)
print_quality_section("Strict containment true but normalized EM false", strict_true_em_false)
print_quality_section("ChartQA relaxed numeric true but normalized EM false", chart_relaxed_em_false)
print_quality_section("ChartQA wrong examples for manual failure-mode labeling", chart_wrong, limit=30)
print_quality_section("Outputs longer than 10 tokens", long_outputs)
print_quality_section("Worst DocVQA/TextVQA ANLS cases", worst_anls)
print_quality_section("Random samples", random_samples, limit=20)


## 17. Fair Method Table And Router Policy

Run `zero_shot`, `shared_doc_text_lora`, `chart_lora_only`, and `hybrid` modes separately to populate reports. This table only compares a backend on tasks it is supposed to serve: ChartQA uses base/chart/hybrid, while DocVQA and TextVQA use shared Doc/Text or hybrid.


In [ ]:
def load_report(path: Path, method: str) -> dict | None:
    if not path.exists():
        return None
    payload = json.loads(path.read_text(encoding="utf-8"))
    return payload.get(method) or payload


def task_report(report: dict, task_type: str) -> dict:
    return report.get("by_task", {}).get(task_type, {})


def metric_from_report(report: dict, task_type: str, metric_name: str) -> float | None:
    return task_report(report, task_type).get(metric_name)


METHOD_TASK_POLICY = {
    "zero_shot": {"chartqa", "docvqa", "textvqa"},
    "shared_lora_all_tasks": {"chartqa", "docvqa", "textvqa"},
    "shared_doc_text_lora": {"docvqa", "textvqa"},
    "docvqa_lora_only": {"docvqa"},
    "textvqa_lora_only": {"textvqa"},
    "chart_lora_only": {"chartqa"},
    "chart_dora_only": {"chartqa"},
    "hybrid_chart_doc_text": {"chartqa", "docvqa", "textvqa"},
    "debug_overfit": {"docvqa", "textvqa"},
}


def method_serves_task(method: str, task_type: str) -> bool:
    if method in METHOD_TASK_POLICY:
        return task_type in METHOD_TASK_POLICY[method]
    if method.startswith("chart_dora_"):
        return task_type == "chartqa"
    if method.startswith("docvqa_dora_"):
        return task_type == "docvqa"
    if method.startswith("textvqa_dora_"):
        return task_type == "textvqa"
    return False


def task_main_score(report: dict, task_type: str) -> float | None:
    if task_type == "chartqa":
        return metric_from_report(report, task_type, "chart_hybrid_accuracy")
    if task_type == "docvqa":
        return metric_from_report(report, task_type, "docvqa_anls") or metric_from_report(report, task_type, "anls") or metric_from_report(report, task_type, "normalized_exact_match")
    if task_type == "textvqa":
        return metric_from_report(report, task_type, "textvqa_vqa_score") or metric_from_report(report, task_type, "vqa_score") or metric_from_report(report, task_type, "normalized_exact_match")
    return metric_from_report(report, task_type, "normalized_exact_match")


def fmt(value: float | None) -> str:
    return "   -  " if value is None else f"{value:.4f}"


METHOD_REPORT_PATHS = {
    "zero_shot": report_path_for_method("zero_shot"),
    "shared_lora_all_tasks": report_path_for_method("shared_lora_all_tasks"),
    "shared_doc_text_lora": report_path_for_method("shared_doc_text_lora"),
    "docvqa_lora_only": report_path_for_method("docvqa_lora_only"),
    "textvqa_lora_only": report_path_for_method("textvqa_lora_only"),
    "chart_lora_only": report_path_for_method("chart_lora_only"),
    "chart_dora_only": report_path_for_method("chart_dora_only"),
    "hybrid_chart_doc_text": report_path_for_method("hybrid_chart_doc_text"),
    "debug_overfit": report_path_for_method("debug_overfit"),
}
for task_name, configs in TASK_DORA_SWEEP_CONFIGS.items():
    for config in configs:
        sweep_method = f"{task_name}_dora_{config['name']}"
        METHOD_REPORT_PATHS[sweep_method] = report_path_for_method(sweep_method)

reports_by_method = {}
for method, path in METHOD_REPORT_PATHS.items():
    report = load_report(path, method)
    if report:
        reports_by_method[method] = report
reports_by_method[method_name] = quality_report

print("Method                    Overall  ChartHy  DocEM   DocANLS TextEM  TextVQA TextF1  AvgLen  >10Tok")
for method in [
    "zero_shot",
    "shared_lora_all_tasks",
    "shared_doc_text_lora",
    "docvqa_lora_only",
    "textvqa_lora_only",
    *[f"docvqa_dora_{config['name']}" for config in TASK_DORA_SWEEP_CONFIGS["docvqa"]],
    *[f"textvqa_dora_{config['name']}" for config in TASK_DORA_SWEEP_CONFIGS["textvqa"]],
    "chart_lora_only",
    "chart_dora_only",
    *[f"chart_dora_{config['name']}" for config in TASK_DORA_SWEEP_CONFIGS["chartqa"]],
    "hybrid_chart_doc_text",
    "debug_overfit",
]:
    report = reports_by_method.get(method)
    if not report:
        continue
    overall = report["overall"]
    chart_hybrid = metric_from_report(report, "chartqa", "chart_hybrid_accuracy") if method_serves_task(method, "chartqa") else None
    doc_em = metric_from_report(report, "docvqa", "normalized_exact_match") if method_serves_task(method, "docvqa") else None
    doc_anls = metric_from_report(report, "docvqa", "docvqa_anls") if method_serves_task(method, "docvqa") else None
    text_em = metric_from_report(report, "textvqa", "normalized_exact_match") if method_serves_task(method, "textvqa") else None
    text_vqa = metric_from_report(report, "textvqa", "textvqa_vqa_score") if method_serves_task(method, "textvqa") else None
    text_f1 = metric_from_report(report, "textvqa", "token_f1") if method_serves_task(method, "textvqa") else None
    print(
        f"{method:<25} "
        f"{overall.get('normalized_exact_match', 0):.4f}  "
        f"{fmt(chart_hybrid)}  "
        f"{fmt(doc_em)}  "
        f"{fmt(doc_anls)}  "
        f"{fmt(text_em)}  "
        f"{fmt(text_vqa)}  "
        f"{fmt(text_f1)}  "
        f"{overall.get('avg_output_token_length', 0):.2f}  "
        f"{overall.get('pct_outputs_gt_10_tokens', 0):.3f}"
    )

def best_backend_for_task(task_type: str) -> tuple[str, float] | None:
    scored = []
    for method, report in reports_by_method.items():
        if not method_serves_task(method, task_type):
            continue
        score = task_main_score(report, task_type)
        if score is not None:
            scored.append((method, score))
    scored = sorted(scored, key=lambda item: item[1], reverse=True)
    return scored[0] if scored else None


print("\nRouter policy from best available task metric:")
for task_type in ["chartqa", "docvqa", "textvqa"]:
    best = best_backend_for_task(task_type)
    if best is None:
        print(f"{task_type} -> no report yet")
        continue
    print(f"{task_type} -> {best[0]}")

print("\nBest eligible backend by task:")
for task_type in ["chartqa", "docvqa", "textvqa"]:
    best = best_backend_for_task(task_type)
    if best is None:
        continue
    best_method, best_score = best
    metric_name = {
        "chartqa": "chart_hybrid_accuracy",
        "docvqa": "docvqa_anls / normalized EM fallback",
        "textvqa": "textvqa_vqa_score / normalized EM fallback",
    }[task_type]
    print(f"{task_type}: {best_method} ({metric_name}={best_score:.4f})")


In [ ]:
def load_quality_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def paired_key(record: dict) -> tuple:
    answers = tuple(str(answer) for answer in (record.get("answers") or []))
    return (
        record.get("task_type"),
        record.get("question"),
        answers,
    )


def paired_task_score(record: dict, task_type: str) -> float:
    if task_type == "chartqa":
        return float(record.get("chart_hybrid_accuracy") or 0.0)
    if task_type == "docvqa":
        return float(record.get("docvqa_anls") or record.get("anls") or record.get("normalized_exact_match") or 0.0)
    if task_type == "textvqa":
        return float(record.get("textvqa_vqa_score") or record.get("vqa_score") or record.get("normalized_exact_match") or 0.0)
    return float(record.get("normalized_exact_match") or 0.0)


def paired_compare_with_zero(method: str, task_type: str, limit: int = 15) -> None:
    zero_rows = [row for row in load_quality_jsonl(prediction_path_for_method("zero_shot")) if row.get("task_type") == task_type]
    method_rows = [row for row in load_quality_jsonl(prediction_path_for_method(method)) if row.get("task_type") == task_type]
    zero_by_key = {paired_key(row): row for row in zero_rows}
    method_by_key = {paired_key(row): row for row in method_rows}
    keys = [key for key in zero_by_key if key in method_by_key]
    if not keys:
        print(f"No paired rows for zero_shot vs {method} on {task_type}.")
        print(f"zero rows={len(zero_rows)} {method} rows={len(method_rows)}")
        return

    cases = []
    for key in keys:
        zero = zero_by_key[key]
        candidate = method_by_key[key]
        zero_score = paired_task_score(zero, task_type)
        method_score = paired_task_score(candidate, task_type)
        if method_score > zero_score:
            case = "adapter_better"
        elif zero_score > method_score:
            case = "zero_better"
        else:
            case = "tie"
        cases.append((case, zero_score, method_score, zero, candidate))

    counts = Counter(case for case, *_ in cases)
    zero_avg = sum(item[1] for item in cases) / len(cases)
    method_avg = sum(item[2] for item in cases) / len(cases)
    print(f"\nPaired compare: zero_shot vs {method} on {task_type}")
    print(f"paired rows: {len(cases)}")
    print(f"zero avg:   {zero_avg:.4f}")
    print(f"{method} avg: {method_avg:.4f}")
    print("cases:", dict(counts))

    def show(case_name: str) -> None:
        subset = [item for item in cases if item[0] == case_name]
        print(f"\n## {case_name} ({len(subset)})")
        for _, zero_score, method_score, zero, candidate in subset[:limit]:
            print("Q:", zero.get("question"))
            print("A:", zero.get("answers"))
            print(f"zero   score={zero_score:.4f} pred={zero.get('cleaned_prediction')!r}")
            print(f"{method:<7} score={method_score:.4f} pred={candidate.get('cleaned_prediction')!r}")
            print()

    show("adapter_better")
    show("zero_better")


PAIRED_COMPARE_METHODS = [
    ("docvqa_lora_only", "docvqa"),
    ("textvqa_lora_only", "textvqa"),
    *[(f"docvqa_dora_{config['name']}", "docvqa") for config in TASK_DORA_SWEEP_CONFIGS["docvqa"]],
    *[(f"textvqa_dora_{config['name']}", "textvqa") for config in TASK_DORA_SWEEP_CONFIGS["textvqa"]],
    ("shared_doc_text_lora", "docvqa"),
    ("shared_doc_text_lora", "textvqa"),
    ("chart_dora_only", "chartqa"),
    *[(f"chart_dora_{config['name']}", "chartqa") for config in TASK_DORA_SWEEP_CONFIGS["chartqa"]],
]

for method, task_type in PAIRED_COMPARE_METHODS:
    if prediction_path_for_method(method).exists() and prediction_path_for_method("zero_shot").exists():
        paired_compare_with_zero(method, task_type, limit=10)
